In [6]:
# =========================
# final-project-python/_4_intraday_backfill.ipynb  (ONE CELL)
# Intraday: initial + missing-first (calendar) + backoff
# FIX: reuse ONE SQLAlchemy ENGINE (prevents "too many clients")
# Includes Yahoo fetch symbol convert "." -> "-"
# Runs ALL intraday intervals from config
# =========================

import time
import random
import yfinance as yf
import pandas as pd
import pandas_market_calendars as mcal

from datetime import datetime, timedelta, timezone
from sqlalchemy import text

%run _1_schema.ipynb
%run _6_config_intervals.ipynb

NYSE = mcal.get_calendar("NYSE")

try:
    if "ENGINE" in globals() and ENGINE is not None:
        ENGINE.dispose()
except Exception:
    pass
ENGINE = get_engine()


def yahoo_symbol(symbol: str) -> str:
    return symbol.replace(".", "-").strip()


def load_active_symbols_from_db() -> list[str]:
    with ENGINE.connect() as conn:
        rows = conn.execute(text("SELECT symbol FROM stocks WHERE status = true ORDER BY symbol")).fetchall()
    return [r[0] for r in rows]


def _sleep_backoff(attempt: int, base_sec: float = 2.0, cap_sec: float = 60.0):
    delay = min(cap_sec, base_sec * (2 ** attempt))
    delay = delay * (0.9 + 0.2 * random.random())
    time.sleep(delay)


def to_epoch_sec(series):
    return (pd.to_datetime(series, utc=True).astype("int64") // 10**9)


def _flatten_yf_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    return df


def _interval_seconds(interval: str) -> int:
    s = interval.strip().lower()
    if s.endswith("m"):
        return int(s[:-1]) * 60
    if s.endswith("h"):
        return int(s[:-1]) * 3600
    if s in ("1d", "d"):
        return 86400
    raise ValueError(f"Unsupported interval: {interval}")


def insert_stock_ohlc(df: pd.DataFrame) -> int:
    if df is None or df.empty:
        return 0

    cols = list(df.columns)
    insert_sql = f"""
    INSERT INTO stock_ohlc ({', '.join(cols)})
    VALUES ({', '.join([f":{c}" for c in cols])})
    ON CONFLICT ON CONSTRAINT uq_symbol_datatype_ts DO NOTHING
    """
    records = df.to_dict(orient="records")

    with ENGINE.begin() as conn:
        conn.execute(text(insert_sql), records)

    return len(records)


def count_rows(symbol: str, data_type: str, ts_from: int, ts_to: int) -> int:
    sql = """
    SELECT COUNT(1)
    FROM stock_ohlc
    WHERE symbol = :symbol
      AND data_type = :data_type
      AND ts BETWEEN :ts_from AND :ts_to
    """
    with ENGINE.connect() as conn:
        return int(conn.execute(text(sql), {
            "symbol": symbol, "data_type": data_type, "ts_from": ts_from, "ts_to": ts_to
        }).scalar() or 0)


def _last_trading_schedule(n_sessions: int, lookback_days: int = 180) -> pd.DataFrame:
    end = pd.Timestamp.utcnow()
    start = end - pd.Timedelta(days=lookback_days)
    sched = NYSE.schedule(start_date=start, end_date=end)
    return sched.tail(n_sessions)


def fetch_intraday_window(symbol: str, interval: str, start_dt: datetime, end_dt: datetime,
                          filter_ts_from: int | None = None, filter_ts_to: int | None = None) -> pd.DataFrame | None:
    sym_fetch = yahoo_symbol(symbol)

    df = yf.download(sym_fetch, interval=interval, start=start_dt, end=end_dt, progress=False)
    if df is None or df.empty:
        return None

    df = _flatten_yf_columns(df).reset_index()
    dt_col = df.columns[0]

    df["ts"] = to_epoch_sec(df[dt_col])
    df["symbol"] = symbol      # store original symbol (e.g. BRK.B)
    df["data_type"] = interval

    df = df.rename(columns={"Open":"open","High":"high","Low":"low","Close":"close","Volume":"volume"})
    df = df[["symbol","data_type","ts","open","high","low","close","volume"]]
    df = df.dropna(subset=["open", "high", "low", "close"])

    if filter_ts_from is not None and filter_ts_to is not None:
        df = df[(df["ts"] >= filter_ts_from) & (df["ts"] < filter_ts_to)]

    return None if df.empty else df


def _chunk_ranges(start: datetime, end: datetime, max_days: int):
    cur = start
    while cur < end:
        nxt = min(cur + timedelta(days=max_days), end)
        yield cur, nxt
        cur = nxt


def initial_intraday_backfill(symbols, interval="1m", sessions: int | None = None,
                              max_attempts: int = 3, base_sleep_sec: float = 0.02, sleep_sec: float = 0.02):
    if sessions is None:
        sessions = int(MAIN_TABLE_RETENTION_DAYS.get(interval, 7))

    api_max_days = int(API_MAX_DAYS.get(interval, 7))
    sched = _last_trading_schedule(sessions)

    open_ts = int(pd.Timestamp(sched.iloc[0]["market_open"]).tz_convert("UTC").timestamp())
    close_ts = int(pd.Timestamp(sched.iloc[-1]["market_close"]).tz_convert("UTC").timestamp())
    start_dt = datetime.fromtimestamp(open_ts, tz=timezone.utc)
    end_dt = datetime.fromtimestamp(close_ts, tz=timezone.utc)

    total = 0
    for sym in symbols:
        sym = sym.strip()
        if not sym:
            continue

        print(f"[intraday initial] {sym} interval={interval} sessions={sessions}")

        for a, b in _chunk_ranges(start_dt, end_dt, api_max_days):
            for attempt in range(max_attempts):
                try:
                    df = fetch_intraday_window(sym, interval, a, b)
                    total += insert_stock_ohlc(df)
                    break
                except Exception as e:
                    print(f"[intraday initial] {sym} {a.date()}..{b.date()} attempt={attempt+1}/{max_attempts} error={e}")
                    if attempt < max_attempts - 1:
                        _sleep_backoff(attempt, base_sec=base_sleep_sec)
                    else:
                        print(f"[intraday initial] {sym} chunk give up for now")
            time.sleep(sleep_sec)

        print(f"[intraday initial] {sym} done")

    print("[intraday initial] Total inserted:", total)


def backfill_intraday_missing(symbols, interval="1m", sessions: int | None = None,
                              coverage_ratio: float = 0.9,
                              max_attempts: int = 3, base_sleep_sec: float = 0.02, sleep_sec: float = 0.02):
    if sessions is None:
        sessions = int(MAIN_TABLE_RETENTION_DAYS.get(interval, 7))

    step = _interval_seconds(interval)
    sched = _last_trading_schedule(sessions)

    total = 0
    for sym in symbols:
        sym = sym.strip()
        if not sym:
            continue

        print(f"[intraday missing] {sym} interval={interval} sessions={sessions}")

        for day, row in sched.iterrows():
            open_ts = int(pd.Timestamp(row["market_open"]).tz_convert("UTC").timestamp())
            close_ts = int(pd.Timestamp(row["market_close"]).tz_convert("UTC").timestamp())

            expected = max(1, (close_ts - open_ts) // step)
            got = count_rows(sym, interval, open_ts, close_ts - 1)

            if got >= int(expected * coverage_ratio):
                continue

            start_dt = datetime.fromtimestamp(open_ts, tz=timezone.utc)
            end_dt = datetime.fromtimestamp(close_ts, tz=timezone.utc)

            for attempt in range(max_attempts):
                try:
                    df = fetch_intraday_window(sym, interval, start_dt, end_dt,
                                               filter_ts_from=open_ts, filter_ts_to=close_ts)
                    total += insert_stock_ohlc(df)
                    break
                except Exception as e:
                    print(f"[intraday missing] {sym} {day.date()} attempt={attempt+1}/{max_attempts} error={e}")
                    if attempt < max_attempts - 1:
                        _sleep_backoff(attempt, base_sec=base_sleep_sec)
                    else:
                        print(f"[intraday missing] {sym} {day.date()} give up for now")

            time.sleep(sleep_sec)

        print(f"[intraday missing] {sym} done")

    print("[intraday missing] Total inserted:", total)


# -----------------------
# RUN SESSION
# -----------------------
symbols = load_active_symbols_from_db()
print("active symbols:", len(symbols))

RUN_MODE = "initial"  # "initial" or "missing"

intervals = [k for k in MAIN_TABLE_RETENTION_DAYS.keys() if k != "1d"]

for interval in intervals:
    print("\n==============================")
    print("RUN interval:", interval, "mode:", RUN_MODE)
    print("==============================")

    if RUN_MODE == "initial":
        initial_intraday_backfill(symbols, interval=interval, sessions=int(MAIN_TABLE_RETENTION_DAYS[interval]))
    else:
        backfill_intraday_missing(symbols, interval=interval, sessions=int(MAIN_TABLE_RETENTION_DAYS[interval]), coverage_ratio=0.95)


active symbols: 503

RUN interval: 1m mode: initial
[intraday initial] A interval=1m sessions=14


C:\Users\lamwi\AppData\Local\Temp\ipykernel_39244\2507481606.py:102: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow()


[intraday initial] A done
[intraday initial] AAPL interval=1m sessions=14
[intraday initial] AAPL done
[intraday initial] ABBV interval=1m sessions=14
[intraday initial] ABBV done
[intraday initial] ABNB interval=1m sessions=14
[intraday initial] ABNB done
[intraday initial] ABT interval=1m sessions=14
[intraday initial] ABT done
[intraday initial] ACGL interval=1m sessions=14
[intraday initial] ACGL done
[intraday initial] ACN interval=1m sessions=14
[intraday initial] ACN done
[intraday initial] ADBE interval=1m sessions=14
[intraday initial] ADBE done
[intraday initial] ADI interval=1m sessions=14
[intraday initial] ADI done
[intraday initial] ADM interval=1m sessions=14
[intraday initial] ADM done
[intraday initial] ADP interval=1m sessions=14
[intraday initial] ADP done
[intraday initial] ADSK interval=1m sessions=14
[intraday initial] ADSK done
[intraday initial] AEE interval=1m sessions=14
[intraday initial] AEE done
[intraday initial] AEP interval=1m sessions=14
[intraday initi

C:\Users\lamwi\AppData\Local\Temp\ipykernel_39244\2507481606.py:102: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow()
$A: possibly delisted; no price data found  (5m 2025-12-04 14:30:00+00:00 -> 2026-02-02 14:30:00+00:00) (Yahoo error = "5m data not available for startTime=1764858600 and endTime=1770042600. The requested range must be within the last 60 days.")

1 Failed download:
['A']: possibly delisted; no price data found  (5m 2025-12-04 14:30:00+00:00 -> 2026-02-02 14:30:00+00:00) (Yahoo error = "5m data not available for startTime=1764858600 and endTime=1770042600. The requested range must be within the last 60 days.")


[intraday initial] A done
[intraday initial] AAPL interval=5m sessions=60


...

[intraday initial] ZTS done
[intraday initial] Total inserted: 52312

RUN interval: 1h mode: initial
[intraday initial] A interval=1h sessions=180
[intraday initial] A done
[intraday initial] AAPL interval=1h sessions=180
[intraday initial] AAPL done
[intraday initial] ABBV interval=1h sessions=180
[intraday initial] ABBV done
[intraday initial] ABNB interval=1h sessions=180
[intraday initial] ABNB done
[intraday initial] ABT interval=1h sessions=180
[intraday initial] ABT done
[intraday initial] ACGL interval=1h sessions=180
[intraday initial] ACGL done
[intraday initial] ACN interval=1h sessions=180
[intraday initial] ACN done
[intraday initial] ADBE interval=1h sessions=180
[intraday initial] ADBE done
[intraday initial] ADI interval=1h sessions=180
[intraday initial] ADI done
[intraday initial] ADM interval=1h sessions=180
[intraday initial] ADM done
[intraday initial] ADP interval=1h sessions=180
[intraday initial] ADP done
[intraday initial] ADSK interval=1h sessions=180
[intrada

C:\Users\lamwi\AppData\Local\Temp\ipykernel_39244\2507481606.py:102: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow()


[intraday initial] A done
[intraday initial] AAPL interval=4h sessions=365
[intraday initial] AAPL done
[intraday initial] ABBV interval=4h sessions=365
[intraday initial] ABBV done
[intraday initial] ABNB interval=4h sessions=365
[intraday initial] ABNB done
[intraday initial] ABT interval=4h sessions=365
[intraday initial] ABT done
[intraday initial] ACGL interval=4h sessions=365
[intraday initial] ACGL done
[intraday initial] ACN interval=4h sessions=365
[intraday initial] ACN done
[intraday initial] ADBE interval=4h sessions=365
[intraday initial] ADBE done
[intraday initial] ADI interval=4h sessions=365
[intraday initial] ADI done
[intraday initial] ADM interval=4h sessions=365
[intraday initial] ADM done
[intraday initial] ADP interval=4h sessions=365
[intraday initial] ADP done
[intraday initial] ADSK interval=4h sessions=365
[intraday initial] ADSK done
[intraday initial] AEE interval=4h sessions=365
[intraday initial] AEE done
[intraday initial] AEP interval=4h sessions=365
[i


1 Failed download:
['EQT']: TypeError("'NoneType' object is not subscriptable")


[intraday initial] EQT done
[intraday initial] ERIE interval=4h sessions=365
[intraday initial] ERIE done
[intraday initial] ES interval=4h sessions=365
[intraday initial] ES done
[intraday initial] ESS interval=4h sessions=365
[intraday initial] ESS done
[intraday initial] ETN interval=4h sessions=365
[intraday initial] ETN done
[intraday initial] ETR interval=4h sessions=365
[intraday initial] ETR done
[intraday initial] EVRG interval=4h sessions=365
[intraday initial] EVRG done
[intraday initial] EW interval=4h sessions=365
[intraday initial] EW done
[intraday initial] EXC interval=4h sessions=365
[intraday initial] EXC done
[intraday initial] EXE interval=4h sessions=365
[intraday initial] EXE done
[intraday initial] EXPD interval=4h sessions=365
[intraday initial] EXPD done
[intraday initial] EXPE interval=4h sessions=365
[intraday initial] EXPE done
[intraday initial] EXR interval=4h sessions=365
[intraday initial] EXR done
[intraday initial] F interval=4h sessions=365
[intraday 